# POC — Coleta de Dados do YouTube (Ceará 2026)

Pipeline de ingestão via **YouTube Data API v3**, conforme o escopo de [`fontes_youtube.md`](../fontes_youtube.md).

- Canais previstos: Ciro Gomes, Elmano de Freitas, O Povo e Diário do Nordeste.
- **Somente os canais ativados em `POC_CHANNELS` são coletados**; as ativações do usuário são preservadas.

Objetivo: coletar todos os vídeos acessíveis dos canais ativos na janela compartilhada `COLLECTION_START`–`COLLECTION_END`, seus comentários e respostas acessíveis, e persistir JSON Bronze local. **Não há filtro de título, nomes, tema ou categoria nesta ingestão**; a seleção temática fica para a futura camada Silver. O fluxo poderá futuramente ser orquestrado pelo Airflow.

## 0. Configuração

Antes de rodar: copie `.env.example` para `.env` na raiz do projeto e preencha `YOUTUBE_API_KEY` com uma chave de API do Google Cloud Console (API "YouTube Data API v3" habilitada). O `.env` já está no `.gitignore` — nunca comitar a chave.

In [1]:
%pip install --quiet google-api-python-client python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os

from dotenv import load_dotenv
from googleapiclient.discovery import build

load_dotenv()

API_KEY = os.environ.get("YOUTUBE_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "YOUTUBE_API_KEY não encontrada. Crie um arquivo .env na raiz do projeto "
        "(veja .env.example) com sua chave da YouTube Data API v3."
    )

youtube = build("youtube", "v3", developerKey=API_KEY)
print("Cliente da YouTube Data API criado com sucesso.")

Cliente da YouTube Data API criado com sucesso.


## 1. Canais da POC

Handles/URLs identificados em `fontes_youtube.md`. Alguns são handles (`@...`), outros ainda usam o formato legado `c/NomeDoCanal` — para esses últimos, resolvemos por busca do nome exato (fallback), já que `forHandle` só funciona com handles `@`.

In [3]:
POC_CHANNELS = {
    "ciro_gomes": {"handle": "CiroGomesOficial", "categoria": "candidato"},
    # "elmano_de_freitas": {"handle": "ElmanodeFreitas", "categoria": "candidato"},
    # "o_povo": {"handle": "opovo", "categoria": "imprensa"},
    # "diario_do_nordeste": {"handle": "diariodonordeste", "categoria": "imprensa"},
}
POC_CHANNELS

{'diario_do_nordeste': {'handle': 'diariodonordeste', 'categoria': 'imprensa'}}

## 2. Resolver `channelId` a partir do handle

A API precisa do `channelId` (formato `UC...`) para as próximas chamadas. Usamos `channels.list(part="id,snippet,contentDetails", forHandle=...)`, que também já retorna o `uploads` playlist ID em `contentDetails.relatedPlaylists.uploads` — economizando uma chamada.

In [4]:
def resolve_channel(handle: str) -> dict | None:
    """Resolve channelId e uploads playlist a partir de um handle (@...)."""
    response = (
        youtube.channels()
        .list(part="id,snippet,contentDetails", forHandle=handle)
        .execute()
    )
    items = response.get("items", [])
    if not items:
        return None
    item = items[0]
    return {
        "channel_id": item["id"],
        "title": item["snippet"]["title"],
        "uploads_playlist_id": item["contentDetails"]["relatedPlaylists"]["uploads"],
    }


for key, info in POC_CHANNELS.items():
    resolved = resolve_channel(info["handle"])
    if resolved is None:
        print(f"[AVISO] Não foi possível resolver o handle @{info['handle']} ({key}). Verifique manualmente.")
        continue
    info.update(resolved)
    print(f"{key}: {resolved['title']} -> {resolved['channel_id']}")

diario_do_nordeste: Diário do Nordeste -> UCMf_wuiFqxdhZI1GVx02mmw


## 3. Estatísticas de vídeo (função reutilizável)

`videos.list` aceita até 50 IDs por chamada (1 unidade de cota). A função divide a lista em lotes de até 50 IDs e reúne as estatísticas em um único dicionário.

In [5]:
def get_video_stats(video_ids: list[str]) -> dict[str, dict]:
    """Busca estatísticas em lotes de até 50 IDs, limite do videos.list."""
    stats = {}
    for start in range(0, len(video_ids), 50):
        batch = video_ids[start:start + 50]
        response = (
            youtube.videos()
            .list(part="statistics,snippet", id=",".join(batch))
            .execute()
        )
        for item in response.get("items", []):
            stats[item["id"]] = {
                "views": int(item["statistics"].get("viewCount", 0)),
                "likes": int(item["statistics"].get("likeCount", 0)),
                "comment_count": int(item["statistics"].get("commentCount", 0)),
                "channel_title": item["snippet"]["channelTitle"],
            }
    return stats

diario_do_nordeste: 15 vídeos encontrados


## 4. Coleta de comentários e persistência Bronze — funções reutilizáveis

`commentThreads.list` percorre todas as páginas de threads, sem limite artificial. As respostas embutidas são deduplicadas por ID e comparadas com `totalReplyCount`: quando faltarem respostas (mesmo se o total for menor ou igual a cinco), `comments.list` percorre todas as páginas daquele comentário. IDs e `parent_id` preservam a ligação entre comentário e resposta; texto, curtidas, contagens e datas não passam por filtro temático.

Por minimização, **não persistimos nome, canal ou avatar do autor de comentários**. Isso não anonimiza o conteúdo: **o texto dos comentários pode conter dados pessoais**. Controle acesso, retenção e compartilhamento dos arquivos; não publique o dataset indiscriminadamente.

A Bronze contém **projeções minimizadas com proveniência, não respostas HTTP brutas completas**. Cada vídeo recebe metadados, janela, contagens, timestamps, comentários, progresso da paginação e erros seguros. Erros guardam apenas status HTTP e códigos `reason` parseados, nunca mensagens completas, URLs de requisição ou credenciais.

As funções abaixo apenas definem o comportamento; não fazem coleta ao executar suas células.

In [9]:
import json
import re
from datetime import datetime, timezone

from googleapiclient.errors import HttpError


QUOTA_REASONS = {
    "quotaExceeded", "dailyLimitExceeded", "dailyLimitExceededUnreg",
    "rateLimitExceeded", "userRateLimitExceeded", "rateLimitExceededUnreg",
}


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def safe_api_error(error):
    """Somente status HTTP e códigos reason; nunca mensagem, URI ou str(error)."""
    status = getattr(getattr(error, "resp", None), "status", None)
    status = status if isinstance(status, int) else None
    reasons = []
    if isinstance(error, HttpError):
        try:
            payload = json.loads(error.content)
            for entry in payload.get("error", {}).get("errors", []):
                reason = entry.get("reason")
                if isinstance(reason, str) and re.fullmatch(r"[A-Za-z][A-Za-z0-9_]{0,79}", reason):
                    reasons.append(reason)
        except (ValueError, TypeError, AttributeError):
            pass
    return {"http_status": status, "reasons": sorted(set(reasons))}


def project_comment(item, parent_id=None, total_reply_count=None):
    """Projeção minimizada, sem campos de identificação do autor; texto intacto."""
    snippet = item["snippet"]
    return {
        "comment_id": item["id"],
        "parent_id": parent_id,
        "text": snippet.get("textOriginal", snippet.get("textDisplay", "")),
        "like_count": snippet.get("likeCount", 0),
        "published_at": snippet.get("publishedAt"),
        "updated_at": snippet.get("updatedAt"),
        "total_reply_count": total_reply_count,
    }


def get_video_comments(client, video_id):
    """Esgota threads e respostas acessíveis, sem limite artificial de páginas.

    Retorna progresso mesmo em falha. O chamador persiste antes de parar por cota.
    O término da paginação não garante um snapshot estável da API.
    """
    started_at = utc_now()
    comments = {}
    thread_pages = reply_pages = 0
    threads_complete = replies_complete = False
    errors = []
    gaps = []
    quota_exhausted = False
    status = "completed"

    def pages(resource, **params):
        token = None
        seen_tokens = set()
        while True:
            response = resource.list(**params, **({"pageToken": token} if token else {})).execute(num_retries=3)
            yield response
            token = response.get("nextPageToken")
            if not token:
                return
            if token in seen_tokens:
                # Não entrar em loop se a API repetir um token.
                raise RuntimeError("Repeated pagination token")
            seen_tokens.add(token)

    try:
        for response in pages(client.commentThreads(), part="snippet,replies", videoId=video_id,
                              maxResults=100, textFormat="plainText"):
            thread_pages += 1
            for thread in response.get("items", []):
                top = thread["snippet"]["topLevelComment"]
                top_id = top["id"]
                comments[top_id] = project_comment(top, total_reply_count=thread["snippet"].get("totalReplyCount", 0))
                for reply in thread.get("replies", {}).get("comments", []):
                    comments[reply["id"]] = project_comment(reply, parent_id=top_id)
        threads_complete = True
        # Compara a quantidade única embutida com totalReplyCount, inclusive totais <= 5.
        tops = [comment for comment in comments.values() if comment["parent_id"] is None]
        embedded_counts = {}
        for comment in comments.values():
            parent = comment["parent_id"]
            if parent is not None:
                embedded_counts[parent] = embedded_counts.get(parent, 0) + 1
        for top in tops:
            parent_id = top["comment_id"]
            expected = top["total_reply_count"]
            if embedded_counts.get(parent_id, 0) >= expected:
                continue
            for response in pages(client.comments(), part="snippet", parentId=parent_id,
                                  maxResults=100, textFormat="plainText"):
                reply_pages += 1
                for reply in response.get("items", []):
                    comments[reply["id"]] = project_comment(reply, parent_id=parent_id)
            actual = sum(comment["parent_id"] == parent_id for comment in comments.values())
            if actual < expected:
                gaps.append({"parent_id": parent_id, "reported": expected, "collected": actual})
        replies_complete = True
        if gaps:
            status = "partial"
    except Exception as error:
        # Inclui erros de transporte, sem serializar seu conteúdo potencialmente sensível.
        safe = safe_api_error(error)
        errors.append(safe)
        quota_exhausted = bool(QUOTA_REASONS.intersection(safe["reasons"])) or safe["http_status"] == 429
        if "commentsDisabled" in safe["reasons"]:
            status = "commentsDisabled"
        else:
            status = "partial" if comments else "error"

    records = list(comments.values())
    top_count = sum(comment["parent_id"] is None for comment in records)
    return {
        "comments": records,
        "counts": {"total": len(records), "top_level": top_count, "replies": len(records) - top_count},
        "collection_status": status,
        "pagination": {"threads_complete": threads_complete, "replies_complete": replies_complete,
                       "completed": threads_complete and replies_complete,
                       "thread_pages": thread_pages, "reply_pages": reply_pages},
        "reply_count_gaps": gaps,
        "quota_exhausted": quota_exhausted,
        "errors": errors,
        "started_at": started_at,
        "finished_at": utc_now(),
    }

In [6]:
import os
import tempfile
from pathlib import Path
from uuid import uuid4

import pandas as pd


VIDEO_COLUMNS = [
    "source_key", "categoria", "video_id", "title", "published_at",
    "views", "likes", "comment_count", "channel_title",
]


def discover_repo_root(cwd=None):
    """Encontra a raiz real a partir da raiz ou de tcc-engdados (sem caminho home fixo)."""
    current = Path(cwd or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "tcc-engdados").is_dir() and (candidate / "fontes_youtube.md").is_file():
            return candidate
    raise FileNotFoundError("Execute a partir do repositório ou de tcc-engdados.")


def atomic_write_json(path, document):
    """Arquivo temporário no mesmo filesystem, flush/fsync e replace atômico."""
    path = Path(path)
    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,
                                         prefix=f".{path.name}.", suffix=".tmp", delete=False) as stream:
            temp_path = Path(stream.name)
            json.dump(document, stream, ensure_ascii=False, indent=2, allow_nan=False)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temp_path, path)
    finally:
        if temp_path is not None:
            temp_path.unlink(missing_ok=True)


def collect_bronze(client, videos_df, collection_start, collection_end, *,
                   bronze_root=None, active_sources=None):
    """Persiste TODO o videos_df da janela; sem filtro de título, tema ou categoria.

    bronze_root permite testes em diretório temporário. Na execução normal, usa
    exclusivamente data/bronze/youtube na raiz descoberta do repositório.
    """
    if collection_start.tzinfo is None or collection_end.tzinfo is None or collection_start > collection_end:
        raise ValueError("Janela inválida: use datas com fuso e início <= fim.")
    if not videos_df.empty:
        if not {"video_id", "source_key", "published_at"}.issubset(videos_df.columns):
            raise ValueError("videos_df deve conter video_id, source_key e published_at.")
        if active_sources is not None and not videos_df["source_key"].isin(active_sources).all():
            raise ValueError("videos_df contém canais inativos; execute novamente a etapa 5.1.")
        dates = pd.to_datetime(videos_df["published_at"], utc=True, errors="coerce")
        if not dates.between(collection_start, collection_end, inclusive="both").all():
            raise ValueError("videos_df contém datas inválidas ou fora da janela; execute novamente a etapa 5.1.")
        # to_json normaliza NaN e escalares pandas para JSON, sem modificar o dataframe.
        columns = [column for column in VIDEO_COLUMNS if column in videos_df.columns]
        selected = videos_df.loc[:, columns].drop_duplicates(subset=["video_id"])
        videos = json.loads(selected.to_json(orient="records", force_ascii=False))
    else:
        videos = []
    if any(not isinstance(video["video_id"], str) or not re.fullmatch(r"[A-Za-z0-9_-]+", video["video_id"])
           for video in videos):
        raise ValueError("video_id inválido para persistência local.")

    root = Path(bronze_root) if bronze_root is not None else discover_repo_root() / "data" / "bronze" / "youtube"
    root.mkdir(parents=True, exist_ok=True)
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "_" + uuid4().hex
    run_dir = root / run_id
    run_dir.mkdir(exist_ok=False)
    window = {"start": collection_start.isoformat(), "end": collection_end.isoformat(),
              "inclusive": True, "applies_to": "video_published_at"}
    manifest = {
        "schema_version": 1, "run_id": run_id, "started_at": utc_now(), "finished_at": None,
        "updated_at": utc_now(), "collection_window": window, "collection_status": "in_progress",
        "selected_video_count": len(videos), "attempted_video_count": 0,
        "completed_video_count": 0, "collected_comment_count": 0, "videos": [],
    }
    documents = []
    # Todas as seleções ficam em disco ANTES da primeira chamada de comentários.
    for video in videos:
        filename = video["video_id"] + ".json"
        document = {
            "schema_version": 1, "run_id": run_id, "source_key": video["source_key"],
            "video": video, "collection_window": window,
            "provenance": {"api": "YouTube Data API v3", "representation": "minimized_projection",
                           "full_raw_http_response": False,
                           "video_endpoints": ["playlistItems.list", "videos.list"],
                           "comment_endpoints": ["commentThreads.list", "comments.list"]},
            "created_at": utc_now(), "updated_at": utc_now(), "started_at": None, "finished_at": None,
            "collection_status": "pending", "errors": [], "quota_exhausted": False,
            "pagination": {"threads_complete": False, "replies_complete": False, "completed": False,
                           "thread_pages": 0, "reply_pages": 0},
            "reply_count_gaps": [], "counts": {"total": 0, "top_level": 0, "replies": 0}, "comments": [],
        }
        atomic_write_json(run_dir / filename, document)
        documents.append(document)
        manifest["videos"].append({"video_id": video["video_id"], "source_key": video["source_key"],
                                   "file": filename, "collection_status": "pending", "counts": document["counts"],
                                   "errors": [], "quota_exhausted": False, "finished_at": None})
    atomic_write_json(run_dir / "manifest.json", manifest)

    for document, entry in zip(documents, manifest["videos"]):
        result = get_video_comments(client, document["video"]["video_id"])
        document.update(result)
        document["updated_at"] = utc_now()
        atomic_write_json(run_dir / entry["file"], document)
        for field in ("collection_status", "counts", "errors", "quota_exhausted", "finished_at"):
            entry[field] = document[field]
        manifest["attempted_video_count"] += 1
        manifest["completed_video_count"] += int(document["collection_status"] == "completed")
        manifest["collected_comment_count"] += document["counts"]["total"]
        manifest["updated_at"] = utc_now()
        if document["quota_exhausted"]:
            manifest["collection_status"] = "stopped_quota"
        atomic_write_json(run_dir / "manifest.json", manifest)
        if document["quota_exhausted"]:
            break  # Progresso salvo; não tenta os vídeos seguintes.

    if manifest["collection_status"] != "stopped_quota":
        manifest["collection_status"] = (
            "completed" if manifest["completed_video_count"] == len(videos) else "completed_with_errors"
        )
    manifest["finished_at"] = utc_now()
    manifest["updated_at"] = utc_now()
    atomic_write_json(run_dir / "manifest.json", manifest)
    return run_dir, manifest

,source_key,video_id,title,published_at,views,likes,comment_count,channel_title
0,diario_do_nordeste,y6CFpnszTo0,Ciro x Elmano: análise e bastidores do debate ...,2026-09-10T21:00:37Z,1451,48,7,Diário do Nordeste
1,diario_do_nordeste,U1zybQ6Mzf8,Walkyria Santos banca projeto de teatro com re...,2026-09-10T16:18:49Z,167,2,0,Diário do Nordeste
2,diario_do_nordeste,UZWm4v1RZ-U,Veja os bastidores do debate PontoPoder entre ...,2026-09-10T15:51:18Z,19853,1197,79,Diário do Nordeste
3,diario_do_nordeste,PsBXPi6xjeQ,"Casal de comerciantes foi morto por engano, em...",2026-09-10T14:17:23Z,5299,319,27,Diário do Nordeste
4,diario_do_nordeste,DsWX60hkrAw,"O que é o SAF, o combustível que pode descarbo...",2026-09-10T13:16:55Z,0,0,0,Diário do Nordeste
5,diario_do_nordeste,1uy8eRk0Ioo,Bastidores do Debate com Ciro e Elmano Para o ...,2026-09-10T02:37:33Z,4438,78,12,Diário do Nordeste
6,diario_do_nordeste,hY-CPjFhob4,Debate Entre Ciro e Elmano Para o Governo Do C...,2026-09-10T02:24:59Z,146167,3078,913,Diário do Nordeste
7,diario_do_nordeste,EzGRQatjXS8,Debate Para o Governo Do Ceará | Diário do Nor...,2026-09-09T23:36:44Z,367730,9229,796,Diário do Nordeste
8,diario_do_nordeste,stU-QrI-jrc,"'Teu coordenador é um calango?', pergunta Elma...",2026-09-09T23:07:15Z,18290,558,94,Diário do Nordeste
9,diario_do_nordeste,bGfmKBQlgp8,Elmano fala sobre entregas do governo e Ciro r...,2026-09-09T22:45:40Z,19421,682,143,Diário do Nordeste


## 5. Pipeline diário — canais ativos, sem filtro temático

1. Listar os vídeos de todos os canais ativos na mesma janela UTC inclusiva, preservando `COLLECTION_START` e `COLLECTION_END`.
2. Usar **`videos_df` diretamente**, sem seleção por título, nomes, tema ou categoria.
3. Persistir antecipadamente metadados de **todos** os vídeos selecionados como `pending`; coletar comentários/respostas e atualizar JSON e manifesto a cada vídeo.

A janela se aplica à **publicação dos vídeos, não às datas dos comentários ou respostas**. Os comentários acessíveis desses vídeos são coletados independentemente de quando foram publicados. Filtragem e tratamento analítico serão feitos na Silver.

### 5.1 Etapa 1 — Janela comum para candidatos e imprensa

Coletamos os vídeos acessíveis pela API publicados de **01/09/2026 às 00:00 UTC até o instante de início da coleta**, inclusive. `COLLECTION_END` é capturado uma vez por execução e usado por todos os canais, assim como `COLLECTION_START`.

A paginação percorre toda a playlist de uploads e filtra pela data de publicação, sem parar no primeiro vídeo antigo: a ordem dos itens pode não coincidir com a ordem de publicação. Isso aumenta as chamadas para canais com histórico extenso, mas evita perder vídeos da janela em páginas posteriores. Vídeos privados, removidos ou não expostos pela API não são garantidos.

In [7]:
COLLECTION_START = datetime(2026, 9, 1, tzinfo=timezone.utc)
# Um único instante final por execução, compartilhado por todos os canais.
COLLECTION_END = datetime.now(timezone.utc)


def list_videos_since(
    uploads_playlist_id: str, since: datetime, until: datetime
) -> list[dict]:
    """Lista vídeos acessíveis na playlist, publicados na janela UTC inclusiva."""
    if since.tzinfo is None or until.tzinfo is None:
        raise ValueError("As datas precisam incluir fuso horário.")
    if since > until:
        raise ValueError("A data inicial não pode ser posterior à final.")
    videos = []
    seen_ids = set()
    request = youtube.playlistItems().list(
        part="snippet,contentDetails", playlistId=uploads_playlist_id, maxResults=50
    )
    while request is not None:
        response = request.execute()
        for item in response.get("items", []):
            published_at_str = item["contentDetails"].get("videoPublishedAt")
            if not published_at_str:
                continue
            published_at = datetime.fromisoformat(published_at_str.replace("Z", "+00:00"))
            if not since <= published_at <= until:
                continue
            video_id = item["contentDetails"]["videoId"]
            if video_id in seen_ids:
                continue
            seen_ids.add(video_id)
            videos.append({
                "video_id": video_id,
                "title": item["snippet"]["title"],
                "published_at": published_at_str,
            })
        # Não interrompe ao encontrar um vídeo antigo: a ordem de publicação
        # pode diferir da ordem dos itens na playlist.
        request = youtube.playlistItems().list_next(request, response)
    return videos


pipeline_videos = []
print(f"Janela UTC para todos os canais: {COLLECTION_START.isoformat()} até {COLLECTION_END.isoformat()}")
for key, info in POC_CHANNELS.items():
    if "uploads_playlist_id" not in info:
        print(f"[AVISO] {key} sem uploads_playlist_id resolvido — rode a seção 2 primeiro.")
        continue
    videos = list_videos_since(
        info["uploads_playlist_id"], since=COLLECTION_START, until=COLLECTION_END
    )
    stats = get_video_stats([v["video_id"] for v in videos])
    for v in videos:
        pipeline_videos.append(
            {"source_key": key, "categoria": info["categoria"], **v, **stats.get(v["video_id"], {})}
        )
    print(f"{key}: {len(videos)} vídeo(s) na janela")

videos_df = pd.DataFrame(pipeline_videos, columns=[
    "source_key", "categoria", "video_id", "title", "published_at",
    "views", "likes", "comment_count", "channel_title",
])
videos_df

Salvo: ..\data\raw\youtube\20260911T001842Z\diario_do_nordeste.json


### 5.2 Bronze local — verificação offline e execução explícita

A próxima célula testa somente mocks e arquivos em diretórios temporários: não cria cliente, não lê credenciais e não chama a API. Pode ser executada após as duas células de funções da seção 4, sem executar configuração ou listagem de vídeos. A célula final, identificada como **EXECUÇÃO LIVE**, é a única desta seção que inicia coleta real; requer o cliente e `videos_df` atualizado pela etapa 5.1.

Persistência atual: `data/bronze/youtube/<timestamp_UTC>_<uuid>/`, na raiz do repositório descoberta a partir do diretório atual (inclusive `tcc-engdados`). **Disco local agora; MinIO futuramente.** Uma pasta exclusiva por execução evita sobrescrever coletas anteriores; cada JSON e o manifesto são substituídos atomicamente via temporário no mesmo diretório. A criação/atualização do conjunto inteiro não é transacional: se houver interrupção entre vídeo e manifesto, o JSON do vídeo é a fonte do progresso mais recente. Não há retomada automática.

- Todos os vídeos e o manifesto são gravados antes da primeira chamada de comentários, incluindo vídeos sem comentários e vídeos que permanecerão `pending`.
- `completed` indica paginação encerrada sem falha observada; `partial` preserva dados obtidos antes de falha ou de divergência na quantidade de respostas; `error` indica falha sem comentários obtidos; `commentsDisabled` é explícito.
- `pagination.completed` registra término da paginação separadamente de `reply_count_gaps`. O manifesto termina como `completed`, `completed_with_errors` ou `stopped_quota`; os não tentados continuam `pending`.
- Requisições de comentários usam `execute(num_retries=3)`, com retentativas limitadas. Falha de cota/limite interrompe toda a execução **depois de salvar o progresso do vídeo**, sem insistir nos seguintes.
- A API só expõe conteúdo acessível às credenciais utilizadas. Comentários/respostas privados, removidos, retidos para moderação ou indisponíveis não são garantidos. O conteúdo pode mudar durante a paginação; deduplicação por ID evita repetição, mas não cria um snapshot consistente. Contagens do vídeo e das threads são observações, não garantia de completude absoluta.

Nenhuma configuração de filtros é necessária e o arquivo de termos da pesquisa não é utilizado ou alterado.

In [8]:
# VERIFICAÇÃO OFFLINE: execute somente após as duas células de funções da seção 4.
# Nenhuma referência ao cliente global youtube ou às credenciais.
def test_bronze_offline():
    from copy import deepcopy
    from types import SimpleNamespace
    from unittest.mock import patch

    class FakeAPI:
        def __init__(self, script, before_execute=None):
            self.script = script
            self.calls = []
            self.before_execute = before_execute

        def commentThreads(self):
            return Resource(self, "threads")

        def comments(self):
            return Resource(self, "replies")

    class Resource:
        def __init__(self, api, endpoint):
            self.api, self.endpoint = api, endpoint

        def list(self, **params):
            api, endpoint = self.api, self.endpoint

            class Request:
                def execute(self, num_retries=0):
                    assert num_retries == 3
                    assert params["textFormat"] == "plainText"
                    if api.before_execute:
                        api.before_execute()
                    key = (endpoint, params.get("videoId", params.get("parentId")), params.get("pageToken"))
                    api.calls.append(key)
                    value = api.script[key]
                    if isinstance(value, Exception):
                        raise value
                    return deepcopy(value)

            return Request()

    class NoStringHttpError(HttpError):
        def __str__(self):
            raise AssertionError("HttpError não pode ser convertido em texto")

    def api_error(reason, status=403):
        return NoStringHttpError(
            SimpleNamespace(status=status, reason="DO_NOT_PERSIST_MESSAGE"),
            json.dumps({"error": {"message": "DO_NOT_PERSIST_MESSAGE",
                                  "errors": [{"reason": reason}]}}).encode(),
            uri="https://invalid.example/DO_NOT_PERSIST_URI",
        )

    def comment(comment_id):
        return {"id": comment_id, "snippet": {
            "textOriginal": " Texto sem tema <b>literal</b> 🙂\nemail fictício: a@example.invalid ",
            "textDisplay": "versão de exibição", "likeCount": 7,
            "publishedAt": "2020-01-01T00:00:00Z", "updatedAt": "2030-01-01T00:00:00Z",
            "authorDisplayName": "OMIT_AUTHOR", "authorChannelId": {"value": "OMIT_CHANNEL"},
            "authorProfileImageUrl": "OMIT_AVATAR",
        }}

    def thread(top_id, total=0, replies=()):
        return {"snippet": {"topLevelComment": comment(top_id), "totalReplyCount": total},
                "replies": {"comments": [comment(reply_id) for reply_id in replies]}}

    # 1. Mais de 20 páginas, duplicatas entre páginas e total <= 5 com respostas faltantes.
    script = {}
    for index in range(23):
        token = None if index == 0 else str(index)
        response = {"items": [thread("top", 3, ("r1", "r1"))]}
        if index < 22:
            response["nextPageToken"] = str(index + 1)
        script[("threads", "v1", token)] = response
    script[("replies", "top", None)] = {"items": [comment("r1"), comment("r2")], "nextPageToken": "next"}
    script[("replies", "top", "next")] = {"items": [comment("r2"), comment("r3")]}
    client = FakeAPI(script)
    result = get_video_comments(client, "v1")
    assert result["collection_status"] == "completed" and result["pagination"]["completed"]
    assert result["pagination"]["thread_pages"] == 23 and result["pagination"]["reply_pages"] == 2
    assert result["counts"] == {"total": 4, "top_level": 1, "replies": 3}
    assert len(client.calls) == 25
    assert {item["comment_id"] for item in result["comments"]} == {"top", "r1", "r2", "r3"}
    top = next(item for item in result["comments"] if item["parent_id"] is None)
    assert top["total_reply_count"] == 3 and top["like_count"] == 7
    assert top["text"] == comment("top")["snippet"]["textOriginal"]
    assert top["published_at"] == "2020-01-01T00:00:00Z" and top["updated_at"] == "2030-01-01T00:00:00Z"
    assert all(item["parent_id"] == "top" for item in result["comments"] if item["comment_id"] != "top")
    assert "OMIT_" not in json.dumps(result)
    assert set(top) == {"comment_id", "parent_id", "text", "like_count", "published_at", "updated_at", "total_reply_count"}

    # 2. Respostas embutidas completas dispensam comments.list; zero comentários é completo.
    embedded = FakeAPI({("threads", "v1", None): {"items": [thread("t", 1, ("r",))]}})
    assert get_video_comments(embedded, "v1")["counts"]["replies"] == 1 and len(embedded.calls) == 1
    empty = get_video_comments(FakeAPI({("threads", "v1", None): {"items": []}}), "v1")
    assert empty["counts"]["total"] == 0 and empty["collection_status"] == "completed"
    display_only = comment("display")
    del display_only["snippet"]["textOriginal"]
    assert project_comment(display_only)["text"] == "versão de exibição"

    # 3. Falha parcial de threads mantém a primeira página e não expõe HttpError/URI.
    partial_script = {("threads", "v1", None): {"items": [thread("t")], "nextPageToken": "p2"},
                      ("threads", "v1", "p2"): api_error("backendError", 503)}
    partial = get_video_comments(FakeAPI(partial_script), "v1")
    assert partial["collection_status"] == "partial" and partial["counts"]["total"] == 1
    assert not partial["pagination"]["completed"] and not partial["quota_exhausted"]
    assert partial["errors"] == [{"http_status": 503, "reasons": ["backendError"]}]
    assert "DO_NOT_PERSIST" not in json.dumps(partial)

    # 4. Falha no meio das respostas mantém respostas embutidas e já paginadas.
    reply_failure_script = {
        ("threads", "v1", None): {"items": [thread("t", 4, ("r1",))]},
        ("replies", "t", None): {"items": [comment("r2")], "nextPageToken": "p2"},
        ("replies", "t", "p2"): api_error("backendError", 500),
    }
    partial_reply = get_video_comments(FakeAPI(reply_failure_script), "v1")
    assert partial_reply["counts"]["replies"] == 2 and partial_reply["collection_status"] == "partial"
    assert partial_reply["pagination"]["threads_complete"] and not partial_reply["pagination"]["replies_complete"]

    # 5. Desabilitados, indisponíveis, transporte e payload de erro inválido.
    disabled = get_video_comments(FakeAPI({("threads", "v1", None): api_error("commentsDisabled")}), "v1")
    assert disabled["collection_status"] == "commentsDisabled" and not disabled["quota_exhausted"]
    failed = get_video_comments(FakeAPI({("threads", "v1", None): api_error("videoNotFound", 404)}), "v1")
    assert failed["collection_status"] == "error" and failed["counts"]["total"] == 0
    transport = get_video_comments(FakeAPI({("threads", "v1", None): OSError("DO_NOT_PERSIST")}), "v1")
    assert transport["errors"] == [{"http_status": None, "reasons": []}]
    malformed = NoStringHttpError(SimpleNamespace(status=502, reason="DO_NOT_PERSIST"), b"not JSON")
    assert safe_api_error(malformed) == {"http_status": 502, "reasons": []}

    # 6. Paginação encerrada com contagem divergente não mascara coleta parcial.
    gap = get_video_comments(FakeAPI({("threads", "v1", None): {"items": [thread("t", 2)]},
                                     ("replies", "t", None): {"items": []}}), "v1")
    assert gap["collection_status"] == "partial" and gap["pagination"]["completed"]
    assert gap["reply_count_gaps"] == [{"parent_id": "t", "reported": 2, "collected": 0}]
    repeated = get_video_comments(FakeAPI({
        ("threads", "v1", None): {"items": [], "nextPageToken": "same"},
        ("threads", "v1", "same"): {"items": [], "nextPageToken": "same"},
    }), "v1")
    assert repeated["collection_status"] == "error" and not repeated["pagination"]["completed"]

    # 7. Cota e limitação de taxa são sinais globais de parada.
    for reason, status in [("quotaExceeded", 403), ("dailyLimitExceeded", 403), ("rateLimitExceeded", 429)]:
        quota = get_video_comments(FakeAPI({("threads", "v1", None): api_error(reason, status)}), "v1")
        assert quota["quota_exhausted"] and quota["collection_status"] == "error"

    # 8–12. Persistência, manifesto, atomicidade, entradas vazias e validações.
    start = datetime(2026, 9, 1, tzinfo=timezone.utc)
    end = datetime(2026, 9, 11, tzinfo=timezone.utc)
    frame = pd.DataFrame([
        {"video_id": video_id, "source_key": "active", "categoria": "imprensa",
         "title": "Receita sem nome de candidato", "published_at": "2026-09-05T00:00:00Z", "likes": float("nan")}
        for video_id in ("v1", "v2", "v3")
    ])
    original_frame = frame.copy(deep=True)
    with tempfile.TemporaryDirectory() as directory:
        root = Path(directory)
        replacements = []
        original_replace = os.replace

        def checked_replace(source, target):
            assert Path(source).parent == Path(target).parent and Path(source).suffix == ".tmp"
            replacements.append(Path(target).name)
            return original_replace(source, target)

        def assert_upfront():
            run = next(root.iterdir())
            manifest = json.loads((run / "manifest.json").read_text())
            assert manifest["selected_video_count"] == 3
            assert all(entry["collection_status"] == "pending" for entry in manifest["videos"])
            for entry in manifest["videos"]:
                pending = json.loads((run / entry["file"]).read_text())
                assert pending["collection_status"] == "pending" and pending["video"]["title"] == "Receita sem nome de candidato"
                assert pending["video"]["likes"] is None and pending["source_key"] == "active"

        quota_script = dict(reply_failure_script)
        quota_script[("replies", "t", "p2")] = api_error("quotaExceeded")
        quota_client = FakeAPI(quota_script, before_execute=assert_upfront)
        with patch.object(os, "replace", side_effect=checked_replace):
            run, manifest = collect_bronze(quota_client, frame, start, end, bronze_root=root, active_sources=["active"])
        assert manifest["collection_status"] == "stopped_quota" and manifest["attempted_video_count"] == 1
        assert manifest["collected_comment_count"] == 3
        assert [entry["collection_status"] for entry in manifest["videos"]] == ["partial", "pending", "pending"]
        saved = json.loads((run / "v1.json").read_text())
        assert saved["counts"]["replies"] == 2 and saved["quota_exhausted"] and saved["finished_at"]
        assert saved["collection_window"]["applies_to"] == "video_published_at"
        assert saved["collection_window"]["start"] == start.isoformat()
        assert saved["collection_window"]["end"] == end.isoformat()
        assert not saved["provenance"]["full_raw_http_response"]
        assert json.loads((run / "manifest.json").read_text()) == manifest
        assert not list(run.glob("*.tmp")) and replacements.count("manifest.json") == 3
        assert "DO_NOT_PERSIST" not in (run / "v1.json").read_text()
        previous = {path.name: path.read_bytes() for path in run.iterdir()}

        # Não-cota permite continuar; manifesto do primeiro vídeo é atualizado antes do segundo.
        def assert_incremental():
            if continuing.calls:
                current_run = next(path for path in root.iterdir() if path != run)
                current_manifest = json.loads((current_run / "manifest.json").read_text())
                assert current_manifest["videos"][0]["collection_status"] == "commentsDisabled"
                assert current_manifest["attempted_video_count"] >= 1

        continuing = FakeAPI({("threads", "v1", None): api_error("commentsDisabled"),
                              ("threads", "v2", None): {"items": []},
                              ("threads", "v3", None): {"items": [thread("last")]}}, assert_incremental)
        second, second_manifest = collect_bronze(continuing, frame, start, end, bronze_root=root)
        assert second != run and second_manifest["attempted_video_count"] == 3
        assert second_manifest["completed_video_count"] == 2 and second_manifest["collection_status"] == "completed_with_errors"
        assert len(continuing.calls) == 3
        assert all(path.read_bytes() == previous[path.name] for path in run.iterdir())

        no_calls = FakeAPI({})
        empty_run, empty_manifest = collect_bronze(no_calls, pd.DataFrame(), start, end, bronze_root=root)
        assert empty_manifest["collection_status"] == "completed" and empty_manifest["selected_video_count"] == 0
        assert empty_manifest["videos"] == [] and not no_calls.calls
        assert [path.name for path in empty_run.iterdir()] == ["manifest.json"]

        duplicate_frame = pd.concat([frame.iloc[:1], frame.iloc[:1]], ignore_index=True)
        _, unique_manifest = collect_bronze(FakeAPI({("threads", "v1", None): {"items": []}}),
                                            duplicate_frame, start, end, bronze_root=root)
        assert unique_manifest["selected_video_count"] == 1 and unique_manifest["collection_status"] == "completed"
        for bad_frame, sources in [(frame, ["inactive"]),
                                   (frame.assign(published_at="2020-01-01T00:00:00Z"), ["active"])]:
            try:
                collect_bronze(no_calls, bad_frame, start, end, bronze_root=root, active_sources=sources)
            except ValueError:
                pass
            else:
                raise AssertionError("Deveria rejeitar dataframe desatualizado")
        pd.testing.assert_frame_equal(frame, original_frame)
    repo = discover_repo_root()
    assert discover_repo_root(repo) == discover_repo_root(repo / "tcc-engdados")
    print("OFFLINE OK — 12 grupos: paginação ilimitada, respostas, dedup, minimização, falhas, cota, Bronze/manifesto e atomicidade.")


test_bronze_offline()

,source_key,videos,views,likes,comentarios
0,diario_do_nordeste,15,697223,18834,2536


In [13]:
# EXECUÇÃO LIVE — faz chamadas reais à API; NÃO faz parte da verificação offline.
# Execute a etapa 5.1 nesta sessão para atualizar videos_df e a janela compartilhada.
if not all(name in globals() for name in ("youtube", "videos_df", "COLLECTION_START", "COLLECTION_END", "POC_CHANNELS")):
    raise RuntimeError("Execute a configuração, resolução de canais e etapa 5.1 antes da coleta Bronze.")

bronze_run_dir, bronze_manifest = collect_bronze(
    youtube,
    videos_df,  # Sem dataframe filtrado intermediário; todos os vídeos da janela.
    COLLECTION_START,
    COLLECTION_END,
    active_sources=POC_CHANNELS.keys(),
)
print(f"Bronze local: {bronze_run_dir}")
print(f"Status: {bronze_manifest['collection_status']}")
print(f"Vídeos selecionados: {bronze_manifest['selected_video_count']}; tentados: {bronze_manifest['attempted_video_count']}")
print(f"Comentários e respostas únicos por vídeo: {bronze_manifest['collected_comment_count']}")

Bronze local: /home/estevam/personal/tcc-pos/data/bronze/youtube/20260912T000837886825Z_c5b77422b0f8450da7f7696917866acf
Status: completed
Vídeos selecionados: 88; tentados: 88
Comentários e respostas únicos por vídeo: 2802
